# Tutorial 3 — LangChain: Chains, RAG and Tools

**Course:** Text & Language Processing / LLM Practical Track
**Format:** Hands-on notebook
**Suggested duration:** 90–120 minutes
**Prerequisite:** Tutorial 0 (Hugging Face Ecosystem Tour)

Tutorials 0–2 ran models locally and looked inside them. This notebook does the opposite: it treats the
model as a component you call over an API, and spends its effort on everything *around* the model — prompts,
memory, retrieval and tools.

By the end you will be able to:

- call a hosted chat model and explain why the API is **stateless**, and what that forces you to do;
- compose prompts, models and parsers into **LCEL chains** with the `|` operator, and say what `|` builds;
- run steps in **parallel** and **branch** between them;
- build a **RAG** pipeline end to end: load, split, embed, store, retrieve, answer;
- define **tools** and get a model to request one.

Tutorial 4 picks up where section 8 stops, and turns a tool request into an agent that acts.

> **Note on running this notebook.** Unlike tutorials 0–2, this one needs an API key and network access.
> Nothing here runs offline.

## 0. Mental model: a chain is a pipe

An LLM call is rarely the whole application. Something has to build the prompt, something has to parse the
answer, and often something has to fetch context first. **LangChain** composes those steps into a pipeline
where data flows one way, each step feeding the next:

```text
input → prompt template → model → output parser → result
```

That shape covers a great deal of real work: summarise, classify, translate, answer-from-context. The
notebook builds up to a full retrieval pipeline on exactly this pattern.

What a pipe cannot do is **come back**. A chain has no way to express "if the answer is unsatisfactory, try
a different tool and think again", because that is a cycle and a pipeline runs once, forwards. Section 8
walks right up to that wall — the model asks for a tool and nothing happens, because nothing in a chain can
run it and loop the result back.

That is the subject of tutorial 4. Here, the path through the program is always known in advance.

## 1. Setup

This notebook uses **Groq** as the model provider: it serves open models (Llama, and others) fast, and has a
free tier. Any other chat provider would work — only the two import lines and the model name would change.

```bash
python -m venv .venv
.venv/Scripts/activate        # Windows
# source .venv/bin/activate   # macOS / Linux

pip install langchain langchain-core langchain-groq langchain-community python-dotenv
pip install langgraph langchain-chroma langchain-huggingface sentence-transformers
```

Then create a `.env` file **next to this notebook**:

```text
GROQ_API_KEY=your-key-here
```

> **Keep the key out of git.** A `.env` file is the standard way to hold credentials, but it only protects
> you if it is never committed. Add `.env` to your `.gitignore` before your first commit. A key pasted
> directly into a notebook cell is worse still: notebooks store their own outputs, so the key travels with
> the file even after you delete the cell.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

assert os.getenv("GROQ_API_KEY"), (
    "GROQ_API_KEY not found. Create a .env file next to this notebook "
    "containing: GROQ_API_KEY=your-key-here"
)
print("API key loaded.")

### Code walkthrough — loading credentials

**`load_dotenv()`** reads the `.env` file and copies its contents into the process environment. It returns
quietly whether or not a file was found, which is why the assertion below it is worth having — otherwise the
first failure you see is an authentication error several cells later, which is far less obvious.

**`os.getenv("GROQ_API_KEY")`** reads the variable back. Note that nothing prints the key itself. Get into
the habit now: a printed key ends up in the notebook's saved output.

`ChatGroq` picks this variable up automatically. You never pass the key as an argument.

## 2. Chat models

A chat model takes a **list of messages** and returns one message. Three roles cover almost everything:
`system` sets standing behaviour, `human` is the user, `assistant` is what the model said previously.

This is the same structure as `apply_chat_template` in tutorial 1, section 9 — LangChain is building that
same formatted string underneath, then sending it to a server.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

model = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

messages = [
    SystemMessage("You are a maths teacher. Answer briefly, and show one line of working."),
    HumanMessage("What is 12 * 7?"),
]

response = model.invoke(messages)

print(type(response).__name__)
print()
print(response.content)

### Code walkthrough — `invoke` and the response object

**`ChatGroq(model=..., temperature=0)`** builds a client, not a model — no weights are downloaded, and the
computation happens on Groq's servers. **`temperature=0`** asks for near-deterministic output, which is what
you want while teaching and testing. Raise it when you want variety.

**`.invoke(messages)`** is the universal LangChain verb. Every component in this notebook — models, prompts,
parsers, chains, graphs — exposes `.invoke()`, which is exactly what makes them composable. Learn it once.

The return value is an **`AIMessage`**, not a string. `.content` holds the text; the object also carries
`response_metadata` (token counts, finish reason) and, later in this notebook, `tool_calls`. Reaching for
`.content` immediately is the most common beginner reflex, and it throws away information you will need in
section 8.

### Before you run

The next cell calls the model twice, asking a follow-up question that depends on the first answer.

The second answer will be confused, because **the API is stateless**. Each `invoke` is an independent HTTP
request. The server keeps nothing between calls: no conversation, no memory, no notion that you spoke a
moment ago.

This surprises people who have used a chat UI, where memory appears automatic. It is not automatic there
either — the interface is resending the whole conversation on every turn. That is the only mechanism there
is, and the next section does it explicitly.

In [ ]:
model.invoke([HumanMessage("My name is Maryam.")])

second = model.invoke([HumanMessage("What is my name?")])
print("Without history:", second.content[:160])

## 3. Memory is a list you maintain

Since the server forgets, the client remembers. "Memory" in LangChain is, at its simplest, a Python list you
append to and resend.

In [ ]:
chat_history = [SystemMessage("You are a concise assistant.")]

for turn in ["My name is Maryam.", "What is my name?"]:
    chat_history.append(HumanMessage(turn))
    reply = model.invoke(chat_history)
    chat_history.append(AIMessage(reply.content))
    print(f"You: {turn}")
    print(f"AI : {reply.content}\n")

print(f"history is now {len(chat_history)} messages — every one is resent on each call")

### Code walkthrough — the loop that creates memory

Each turn does three things: append the human message, invoke on **the whole list**, append the reply. Drop
the third step and the model forgets its own answers while remembering your questions.

Two consequences follow immediately, and they shape every production chat system:

**Cost grows quadratically.** Turn 10 resends turns 1–9. You pay per token, per turn, for the same text
repeatedly — which is why tutorial 1's point about token counts being the billing unit matters here.

**The context window is finite.** Long conversations eventually exceed it. Real systems therefore trim old
turns, summarise them into one message, or retrieve only the relevant past — the last of which is section 7's
subject applied to conversation history.

LangChain offers helpers (`ConversationBufferMemory` and friends) that wrap this pattern, but they are
wrappers around exactly this list. Understanding the list is the durable knowledge.

## 4. Prompt templates

Hard-coded prompt strings do not survive contact with a real application. A template separates the fixed
instruction from the variable input.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Answer in at most {sentences} sentences."),
    ("human", "{question}"),
])

filled = template.invoke({
    "role": "patient physics tutor",
    "sentences": 2,
    "question": "Why is the sky blue?",
})

for m in filled.to_messages():
    print(f"[{m.type}] {m.content}")

### Code walkthrough — templates are runnables too

**`from_messages([...])`** takes `(role, text)` tuples and builds a reusable prompt. `{role}`, `{sentences}`
and `{question}` are placeholders filled at call time.

Note that the template is invoked with **`.invoke()`**, the same verb as the model, and returns a message
list rather than a string. That uniformity is the entire design: because a template and a model share an
interface, one can feed the other without glue code. Section 5 exploits this.

The other constructor you will meet is **`from_template()`**, which builds a single human message rather
than a role-tagged list. Prefer `from_messages` — a system message is the most reliable place to put
standing instructions.

## 5. Chains: the `|` operator

Because every component accepts and returns values through `.invoke()`, they compose. LangChain overloads
`|` to express that composition, and the result is called **LCEL** (LangChain Expression Language).

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = template | model | StrOutputParser()

print(chain.invoke({
    "role": "patient physics tutor",
    "sentences": 2,
    "question": "Why is the sky blue?",
}))

### Code walkthrough — what `|` builds

Read the chain right to left as types: a dict goes into the template, which emits messages; messages go into
the model, which emits an `AIMessage`; the `AIMessage` goes into **`StrOutputParser`**, which pulls out
`.content`. Each step's output type matches the next step's input type, and that is all "composable" means
here.

**`|` builds a `RunnableSequence` object.** It is not magic syntax — it is Python's `__or__` operator,
overloaded. These two lines are the same thing:

```python
chain = template | model | StrOutputParser()
chain = RunnableSequence(template, model, StrOutputParser())
```

Anything can join a chain if it implements the runnable interface, including a plain Python function wrapped
in **`RunnableLambda`**. That is the escape hatch whenever a step is just code.

Chains also inherit `.batch()` for many inputs at once and `.stream()` for token-by-token output, without
any extra work on your part.

In [ ]:
from langchain_core.runnables import RunnableLambda

word_count = RunnableLambda(lambda text: f"{text}\n\n[{len(text.split())} words]")

verbose_chain = template | model | StrOutputParser() | word_count

print(verbose_chain.invoke({
    "role": "historian",
    "sentences": 3,
    "question": "Why did the Library of Alexandria decline?",
}))

## 6. Parallel and branching chains

Two structures cover most non-linear work without needing a graph.

**`RunnableParallel`** runs several branches on the same input and collects the results into a dict. The
branches are independent, so they can run concurrently.

**`RunnableBranch`** picks *one* path by testing conditions in order — an if/elif/else whose condition can
depend on a model's own output.

In [ ]:
from langchain_core.runnables import RunnableParallel

pros = ChatPromptTemplate.from_messages([
    ("system", "You are a product reviewer. Be brief."),
    ("human", "List two advantages of {product}."),
]) | model | StrOutputParser()

cons = ChatPromptTemplate.from_messages([
    ("system", "You are a product reviewer. Be brief."),
    ("human", "List two drawbacks of {product}."),
]) | model | StrOutputParser()

both = RunnableParallel(advantages=pros, drawbacks=cons)

result = both.invoke({"product": "an electric kettle with no temperature control"})
print("ADVANTAGES\n", result["advantages"], "\n")
print("DRAWBACKS\n", result["drawbacks"])

### Code walkthrough — parallel branches

**`RunnableParallel(advantages=..., drawbacks=...)`** names each branch. Both receive the *same* input dict,
and the output is a dict under those names.

The gain is latency, not tokens. Two calls still cost two calls, but they overlap instead of queueing, so the
wall-clock time is roughly one call rather than two. With five branches the difference is substantial.

This is also how you assemble context for a later step: a common idiom is a parallel block that gathers
retrieved documents *and* passes the original question through untouched, feeding both into a final prompt.
Section 7 uses exactly that shape.

In [ ]:
from langchain_core.runnables import RunnableBranch

classify = ChatPromptTemplate.from_messages([
    ("system", "Classify the sentiment as exactly one word: positive, negative, or neutral."),
    ("human", "{feedback}"),
]) | model | StrOutputParser()

thanks = ChatPromptTemplate.from_messages([
    ("human", "Write a two-sentence thank-you for this praise: {feedback}")
]) | model | StrOutputParser()

apology = ChatPromptTemplate.from_messages([
    ("human", "Write a two-sentence apology addressing this complaint: {feedback}")
]) | model | StrOutputParser()

follow_up = ChatPromptTemplate.from_messages([
    ("human", "Ask one clarifying question about this feedback: {feedback}")
]) | model | StrOutputParser()

route = RunnableBranch(
    (lambda x: "positive" in x["sentiment"].lower(), thanks),
    (lambda x: "negative" in x["sentiment"].lower(), apology),
    follow_up,                                    # default
)

pipeline = RunnableParallel(
    sentiment=classify,
    feedback=lambda x: x["feedback"],
) | route

for note in ["The delivery arrived a day early and the packaging was lovely.",
             "It stopped working after two days and support never replied."]:
    print(f"> {note}")
    print(pipeline.invoke({"feedback": note}), "\n")

### Code walkthrough — routing on model output

`RunnableBranch` takes `(condition, runnable)` pairs and one final default. Conditions are tested in order;
the first match wins. A missing default raises at runtime, so always supply one.

The interesting part is the `RunnableParallel` in front. `route` needs **two** things — the classification
and the original feedback — but `classify` only returns the label. The parallel block runs the classifier
while passing the raw input through unchanged, so the branch receives `{"sentiment": ..., "feedback": ...}`.
Threading the original input alongside a derived value is one of the most-used patterns in LCEL.

Note what this costs: two model calls per item, one to classify and one to respond. That is the standard
trade for routing — and a reason to classify with something small and cheap when you can.

## 7. RAG: answering from your own documents

A model knows what was in its training data. **Retrieval-Augmented Generation** fixes that by finding
relevant text at query time and putting it in the prompt.

Five steps, always the same:

```text
load → split → embed → store → retrieve → answer
```

We use a small inline corpus so the cell runs anywhere. For real work, swap the list for `TextLoader`,
`PyPDFLoader`, or `WebBaseLoader` from `langchain_community.document_loaders` — everything downstream is
unchanged.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

raw_docs = [
    Document(page_content=(
        "The K. N. Toosi University of Technology was founded in 1928 and is located in Tehran. "
        "It is one of the oldest engineering universities in Iran, named after the Persian polymath "
        "Nasir al-Din al-Tusi, who worked on astronomy, mathematics and ethics in the 13th century."
    ), metadata={"source": "university-handbook"}),
    Document(page_content=(
        "Byte-level BPE tokenizers never produce an unknown token, because every input is representable "
        "as UTF-8 bytes. The trade-off is that text in non-Latin scripts often needs more tokens per word, "
        "which raises both API cost and effective context consumption."
    ), metadata={"source": "tokenization-notes"}),
    Document(page_content=(
        "A KV cache stores the key and value vectors of previous positions so that generating each new "
        "token does not recompute the whole sequence. It trades memory for speed, and its size grows "
        "linearly with sequence length."
    ), metadata={"source": "internals-notes"}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
store = Chroma.from_documents(chunks, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 2})

hits = retriever.invoke("Why does Persian text cost more to send to an LLM?")
for d in hits:
    print(f"[{d.metadata['source']}] {d.page_content[:90]}...")

### Code walkthrough — the retrieval half

**`RecursiveCharacterTextSplitter`** is the splitter to default to. It tries separators in order —
paragraphs, then lines, then sentences, then words — so it breaks at the largest natural boundary that fits
rather than slicing mid-word. `chunk_size` is in characters, not tokens, so a 300-character chunk is roughly
60–80 English tokens.

**`chunk_overlap=50`** repeats the tail of each chunk at the head of the next, so a sentence spanning a
boundary still appears intact in one of them. Without overlap, retrieval quietly fails on exactly the
sentences that straddle a split.

**`HuggingFaceEmbeddings`** runs *locally* — `all-MiniLM-L6-v2` is a small sentence-transformer that maps
text to 384-dimensional vectors. Only the chat model is remote here. Embedding is cheap and needs no API key.

**`Chroma.from_documents`** embeds every chunk and indexes the vectors. As written this is in-memory and
vanishes with the kernel; pass `persist_directory="./db"` to keep it on disk and skip re-embedding.

**`as_retriever(search_kwargs={"k": 2})`** returns the 2 nearest chunks by cosine similarity. `k` is the
knob that matters most: too small and the answer is missing; too large and the real evidence drowns in
irrelevant context.

Notice the query used no keyword from the matching chunk — no "Persian", no "tokens". Matching is by meaning,
in embedding space, which is what makes this different from search by keyword.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using only the context below. "
     "If the context does not contain the answer, say so — do not guess.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

for q in ["Why does Persian text cost more to send to an LLM?",
          "Who won the 2006 World Cup?"]:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}\n")

### Code walkthrough — the generation half

The dict at the head of the chain is a `RunnableParallel` in shorthand. It runs two branches on the incoming
question: **`retriever | format_docs`** fetches chunks and flattens them into a string, while
**`RunnablePassthrough()`** forwards the question untouched. The prompt then has both `{context}` and
`{question}`.

**The system prompt does real work.** "Use only the context" and "say so — do not guess" are what make the
second question fail *honestly*. Without that instruction the model answers from its training data, and you
have built a system that appears grounded but silently is not. Grounding is enforced by the prompt, not by
the retriever.

Including `[source]` markers in the formatted context is the cheapest form of citation — the model can refer
to them, and you can check. For anything user-facing, return the retrieved documents alongside the answer so
a human can verify the claim.

**What breaks in practice** is retrieval, not generation. If a RAG system gives bad answers, inspect what
`retriever.invoke(query)` returned *before* touching the prompt. Most often the right chunk was never
fetched.

### Exercise 7.1

Add a fourth document about something the model already knows — say, a short factual paragraph about
Tehran — and ask a question it could answer from either source. How would you tell whether the answer came
from your document or from the model's own weights? Then set `k=1` and re-run the Persian-text question.
Does the answer survive?

## 8. Tools: letting the model act

A model cannot know today's date, do reliable arithmetic, or look anything up. **Tools** are functions you
expose so it can request one when it needs it.

The model never executes anything. It emits a structured request — "call `multiply` with `a=17, b=23`" — and
*your* code decides whether to run it. That boundary is the whole security model.

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers together."""
    return a * b

@tool
def current_time() -> str:
    """Return the current local time as HH:MM."""
    from datetime import datetime
    return datetime.now().strftime("%H:%M")

tools = [multiply, current_time]
model_with_tools = model.bind_tools(tools)

reply = model_with_tools.invoke([HumanMessage("What is 17 times 23?")])

print("content:", repr(reply.content))
print("tool_calls:", reply.tool_calls)

### Code walkthrough — the docstring is the interface

**`@tool`** turns a function into something the model can be told about. What gets sent is built from three
things, and all three matter:

- the **function name**;
- the **type hints** (`a: int, b: int`), which become a JSON schema for the arguments;
- the **docstring**, which is the only description of *when* to use it.

A vague docstring is a broken tool. "Multiply two integers together" is a specification; "does maths" is not.
This is prompt engineering wearing a Python costume.

**`.bind_tools(tools)`** returns a new model that advertises them. The original `model` is unchanged.

Look at the output: **`content` is empty** and **`tool_calls` is populated**. The model answered by
*requesting a call*, not by producing text — which is why section 2 warned against reaching for `.content`
reflexively. Here it holds nothing.

Nothing has been executed at this point. Something still has to run the function and hand the result back,
and doing that reliably — including when the model wants a second tool after seeing the first result — is a
loop. That is where chains run out and graphs begin.

## 9. Where this goes next

Section 8 ended with a model that asked to call `multiply` and a program that did nothing about it. Closing
that gap needs three things a chain cannot provide:

1. **Execution** — something has to run the requested function and format the result as a message.
2. **A loop** — the result goes back to the model, which may then ask for another tool.
3. **A stopping rule** — the loop must end, whether the model cooperates or not.

All three are control flow, and control flow is what **LangGraph** adds. Tutorial 4 builds the state
machine, then the agent, starting from the tools defined here.

The division worth remembering: **LCEL handles data flow, LangGraph handles control flow.** Most real
applications use both, with chains as the steps inside a graph.

## Glossary — quick reference

| Term | Meaning |
|---|---|
| **Runnable** | Anything exposing `.invoke()` / `.batch()` / `.stream()`. The interface `\|` composes. |
| **LCEL** | LangChain Expression Language — building chains with the `\|` operator. |
| **`RunnableParallel`** | Runs branches on the same input, returns a dict of results. |
| **`RunnableBranch`** | Picks one path by testing conditions in order; needs a default. |
| **`RunnablePassthrough`** | Forwards its input unchanged — used to carry the original alongside derived values. |
| **`RunnableLambda`** | Wraps a plain Python function so it can join a chain. |
| **Stateless API** | The server keeps nothing between calls; the client resends the conversation each turn. |
| **RAG** | Retrieval-Augmented Generation: fetch relevant text at query time and put it in the prompt. |
| **Chunk overlap** | Repeated text at chunk boundaries so split sentences survive intact in at least one chunk. |
| **Retriever** | A component returning the `k` chunks nearest a query in embedding space. |
| **Tool** | A function exposed to the model, described by its name, type hints and docstring. |
| **`tool_calls`** | Structured call requests on an `AIMessage`. The model requests; your code executes. |

---